<a href="https://colab.research.google.com/github/cangurosorte/dependency-map/blob/main/asset_map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Requirements with exact version
### python install via pip
- dash==2.17.0
- jaal==0.1.7
- pandas==2.2.2

## Import Packages

In [ ]:
import pandas as pd
from jaal import Jaal
#from jaal.datasets import load_got
from dash import html as html


/Users/marcosbelarm/dev_workspace/staticdev/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/marcosbelarm/dev_workspace/staticdev/lib/python3.9/site-packages/dash_bootstrap_components/_table.py:5: UserWarning: 
The dash_html_components package is deprecated. Please replace
`import dash_html_components as html` with `from dash import html`
  import dash_html_components as html


# Variables


In [ ]:
#sz_file_path = "csv/dependecy_report.csv"
sz_file_path = "csv/dependency.csv"
#nodes = "csv/refined_study_2/node_filtred_refined.csv"
remove_string = "Unscanned Device"
tcp_webbrowser_port = 8071

## Pandas dataframe load
Import CSV Dependency File from Migration Center -> Stratozone

In [ ]:
sz_file_csv = pd.read_csv(sz_file_path,usecols=['initiatingDeviceName','receivingDeviceName','receivingPort','connectionCount'],
                          header=0,
                        #dtype={'initiatingDeviceName': 'str', 'receivingDeviceName': 'str', 'receivingPort': int, 'connectionCount': int }
                        )

## Load Auxiliary Data

In [ ]:
nodes_df = pd.read_csv(nodes, header=0)

NameError: name 'nodes' is not defined

In [ ]:
nodes_df.head(3)

,id,app_name,is_inscope,migration_wave,environment
0,P0PSWVFTP01,DMZ FTP,True,Wave 1,PROD
1,P0PSWVFTP02,DMZ FTP,True,Wave 1,PROD
2,P0PSWVPLC01,DMZ - Planilla de Contacto,True,Wave 1,DES


## Port Filter


In [ ]:
filtred_ports = sz_file_csv[(sz_file_csv["receivingPort"] != '22' ) & (sz_file_csv["receivingPort"] != '3389' ) & (sz_file_csv["receivingPort"] != '53' )]

### Remove List in ReceivingDeviceName and initiatingDeviceName

In [ ]:
def remove_from_initiating_and_recieving(substring_to_remove, df):
    filter_dst = df['receivingDeviceName'].str.contains(substring_to_remove)
    filtred_df_dst = df[~filter_dst]
    filter_src = filtred_df_dst['initiatingDeviceName'].str.contains(substring_to_remove)
    filtred_df = filtred_df_dst[~filter_src]
    return filtred_df

In [ ]:
filtred_ports

,initiatingDeviceName,receivingDeviceName,receivingPort,connectionCount
0,ARLWLSAPPP01,Unscanned Device,389,1
1,ARLWLSAPPP01,Unscanned Device,443,1
2,ARLWLSAPPP01,Unscanned Device,1537,1
3,ARLWLSAPPP01,Unscanned Device,443,17
4,ARLWLSAPPP01,Unscanned Device,443,3
...,...,...,...,...
109693,Unscanned Device,INMDESAPP09,3209,1
109694,Unscanned Device,INMDESAPP09,3209,8
109695,Unscanned Device,INMDESAPP09,3209,4
109696,Unscanned Device,SGWLSOLDAPPP03,5561,1


In [ ]:
tmp_df = filtred_ports
remove_list = ["Unscanned Device", "P0PSWVCOM01","P0PSWFADS02","P0PSWVADS01"]


In [ ]:
string1 = "Unscanned Device"
#string2 = "P0PSWVCOM01"

post_filter_df = remove_from_initiating_and_recieving(string1,tmp_df)
#post_filter_df = remove_from_initiating_and_recieving(string2,post_filter_df)


In [ ]:
post_filter_df

,initiatingDeviceName,receivingDeviceName,receivingPort,connectionCount
1472,P0PSWFAPL03,P0PSWVCOM02,5723,160
1600,P0PSWFBCK01,P0PSWVPKI02,80,1
1607,P0PSWFBCK01,P0PSWVPKI02,80,1
1614,P0PSWFBCK01,P0PSWVPKI02,80,1
1623,P0PSWFBCK02,P0PSWVSIG03N,5565,1
...,...,...,...,...
17163,P0PSWVWAC01,P0PSWVPKI02,80,1
17170,P0PSWVWAC01,P0PSWVPKI02,80,1
17243,P0PSWVWCT01,P0PSWFBCK02,5565,1
17335,P0PSWVWCT01,P0PSWFBCK02,5565,1


## Dataframe Column rename
Adjust to better work wi JAAL python Module

In [ ]:
df = post_filter_df.rename(
    columns={
        "initiatingDeviceName": "from",
        "receivingDeviceName": "to",
        "connectionCount": "weight"
    }
)

## Summarization

In [ ]:
df_raw = df.groupby(['from','to'], as_index=False)['weight'].sum()

In [ ]:
df_grouped = df_raw[df_raw['weight'] >= 1 ]

In [ ]:
len(df_grouped)

152

In [ ]:
nodes_df

,id,app_name,is_inscope,migration_wave,environment
0,P0PSWVFTP01,DMZ FTP,True,Wave 1,PROD
1,P0PSWVFTP02,DMZ FTP,True,Wave 1,PROD
2,P0PSWVPLC01,DMZ - Planilla de Contacto,True,Wave 1,DES
3,P0PSWVDNS01,DMZ - Forwarder DNS,True,Wave 0,PROD
4,P0PSWFAPL03,DMZ - Planilla de Contacto,True,Wave 1,DES
5,P0PSWVDNS02,DMZ - Forwarder DNS,True,Wave 0,PROD
6,P0PSWVEDC01,DMZ - EDI,True,Wave 1,PROD
7,P0PSWVDMG01,Documanager,True,Wave 1,PROD
8,P0PSWVSIG08,ArcGis,True,Wave 3,PROD
9,P0PSWVSIG07,ArcGis,True,Wave 3,PROD


In [ ]:
df_grouped["type"] = "Undirected"
df_grouped

,from,to,weight,type
0,P0PSWFAPL03,P0PSWVCOM02,160,Undirected
1,P0PSWFBCK01,P0PSWVPKI02,3,Undirected
2,P0PSWFBCK02,P0PSWVSIG03N,2,Undirected
3,P0PSWFBCK02,P0PSWVSIG04,8,Undirected
4,P0PSWFBCK02,P0PSWVSIG07,6,Undirected
...,...,...,...,...
147,P0PSWVTXT01,P0PSWVRDP03,20,Undirected
148,P0PSWVTXT01,P0PSWVSAP05,39,Undirected
149,P0PSWVTXT01,P0PSWVSAP06,6,Undirected
150,P0PSWVWAC01,P0PSWVPKI02,3,Undirected


## Plot
The default method was adjusted doe the number of interactions to stabilize.

In [ ]:
#Jaal(edge_df=df_grouped,node_df=nodes_df).plot(port=8085,directed=False, vis_opts={'height': '1290px','weight': '1100px','interaction':{'hover': True},'physics':{'solver' : 'repulsion'}})
# Jaal(edge_df=df_grouped).plot(port=8085)
edge_df = df_grouped
node_df = nodes_df
Jaal(edge_df=edge_df,node_df=node_df).plot(port=tcp_webbrowser_port,directed=False, vis_opts={'height': '1290px','weight': '1100px','interaction':{'hover': True},'physics':{'solver' : 'repulsion'}})

Parsing the data...Done


No trigger
No trigger
Modifying edge size using  weight
inside color node migration_wave
No trigger
